# Query-guided video frame retrieval

Choose **Runtime → Change runtime type → GPU** before running the setup cell. Upload `query_guided_video_retrieval_colab.zip` when prompted, authorize Drive, and provide the folder containing your videos. The archive is copied to MyDrive for reuse in later sessions.

The interface samples one frame per second, ranks samples with SigLIP2, and displays the similarity timeline and frame gallery. Selected frames can be exported with timestamps. The cost panel is an editable estimate for a downstream VLM request; its assumptions are shown in the interface. Gemini query expansion is optional.

In [ ]:
# Set Runtime > Change runtime type > GPU before running this cell.

import getpass
import html
import os
import shutil
import socket
import subprocess
import sys
import threading
import time
import zipfile
from pathlib import Path

import torch
from google.colab import drive, files, output
from IPython.display import HTML, display
import requests

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is attached to this Colab runtime. Choose Runtime > Change runtime type > GPU, "
        "reconnect, then run this cell again."
    )
print(f"Colab GPU: {torch.cuda.get_device_name(0)}")


def launch_retrieval_app():
    # Avoid starting a duplicate server if this cell is run a second time.
    existing_thread = globals().get("_retrieval_app_server_thread")
    if existing_thread is not None and existing_thread.is_alive():
        port = globals().get("_retrieval_app_port", 8000)
        ui_url = output.eval_js(f"google.colab.kernel.proxyPort({port})")
        if not ui_url.endswith("/"):
            ui_url += "/"
        print("The retrieval app is already running in this Colab runtime.")
        display(HTML(f'<a href="{html.escape(ui_url, quote=True)}" target="_blank" rel="noreferrer" style="font-size:18px;font-weight:700">Open research interface</a>'))
        return

    drive.mount("/content/drive")

    # Keep this ZIP in MyDrive to avoid uploading it again next time.
    app_zip = Path("/content/drive/MyDrive/query_guided_video_retrieval_colab.zip")
    if not app_zip.is_file():
        print("Upload the downloaded query_guided_video_retrieval_colab.zip when the file picker opens.")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if not zip_names:
            raise FileNotFoundError("Please upload query_guided_video_retrieval_colab.zip.")
        uploaded_zip = Path("/content") / zip_names[0]
        app_zip.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(uploaded_zip, app_zip)

    app_dir = Path("/content/query_guided_video_retrieval")
    if app_dir.exists():
        shutil.rmtree(app_dir)
    app_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(app_zip) as bundle:
        for member in bundle.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in app ZIP: {member.filename}")
        bundle.extractall(app_dir)
    if not (app_dir / "app.py").is_file():
        raise FileNotFoundError("The app ZIP does not contain app.py at its root.")

    # Colab already provides CUDA-enabled PyTorch. Install the app dependencies
    # without replacing that GPU build.
    packages = [
        "fastapi>=0.115,<1.0",
        "uvicorn[standard]>=0.30,<1.0",
        "python-dotenv>=1.0,<2.0",
        "requests>=2.32,<3.0",
        "numpy>=1.26,<3.0",
        "opencv-python-headless>=4.10,<5.0",
        "Pillow>=10.0,<12.0",
        "transformers>=4.50,<6.0",
        "safetensors>=0.4,<1.0",
    ]
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *packages],
        check=True,
    )
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA became unavailable after dependency installation; restart the GPU runtime and retry.")

    candidates = [
        Path("/content/drive/MyDrive/UCA_fixed_test_set/videos"),
        Path("/content/drive/MyDrive/ucf_crimes"),
        Path("/content/drive/MyDrive/videos"),
    ]
    default_video_dir = next((path for path in candidates if path.is_dir()), None)
    path_hint = str(default_video_dir) if default_video_dir else "/content/drive/MyDrive/path_to_your_videos"
    video_dir = input(f"Drive folder containing your videos [{path_hint}]: ").strip() or path_hint
    if not Path(video_dir).expanduser().is_dir():
        raise FileNotFoundError(f"Video folder not found: {video_dir}")

    # Put your fine-tuned checkpoint in Drive first, then enter its path here.
    default_checkpoint = "/content/drive/MyDrive/model.safetensors"
    default_model = default_checkpoint if Path(default_checkpoint).is_file() else "google/siglip2-base-patch16-224"
    siglip_model = input(f"SigLIP model or checkpoint path [{default_model}]: ").strip() or default_model
    if siglip_model.lower().endswith(".safetensors") and not Path(siglip_model).is_file():
        raise FileNotFoundError(f"Fine-tuned weights not found: {siglip_model}. Upload them to Drive and enter that path.")
    gemini_api_key = getpass.getpass("Gemini API key (press Enter to skip query rewriting): ").strip()
    os.environ["VIDEO_DIR"] = str(Path(video_dir).expanduser())
    os.environ["SIGLIP_MODEL"] = siglip_model
    os.environ["SIGLIP_BASE_MODEL"] = "google/siglip2-base-patch16-224"
    os.environ["GEMINI_MODEL"] = "gemini-2.5-flash"
    os.environ.setdefault("FRAME_BATCH_SIZE", "16")
    if gemini_api_key:
        os.environ["GEMINI_API_KEY"] = gemini_api_key
    else:
        os.environ.pop("GEMINI_API_KEY", None)

    os.chdir(app_dir)
    sys.path.insert(0, str(app_dir))
    import importlib
    if "app" in sys.modules:
        del sys.modules["app"]
    importlib.invalidate_caches()
    import app as retrieval_module
    import uvicorn

    # Choose port 8000 unless it is already occupied.
    port = 8000
    with socket.socket() as probe:
        try:
            probe.bind(("0.0.0.0", port))
        except OSError:
            with socket.socket() as fallback:
                fallback.bind(("0.0.0.0", 0))
                port = fallback.getsockname()[1]

    server = uvicorn.Server(
        uvicorn.Config(
            retrieval_module.app,
            host="0.0.0.0",
            port=port,
            log_level="warning",
            access_log=False,
        )
    )
    server_thread = threading.Thread(target=server.run, daemon=True, name="retrieval-web")
    server_thread.start()
    globals()["_retrieval_app_server"] = server
    globals()["_retrieval_app_server_thread"] = server_thread
    globals()["_retrieval_app_port"] = port

    for attempt in range(60):
        try:
            config = requests.get(f"http://127.0.0.1:{port}/api/config", timeout=2).json()
            break
        except Exception:
            if attempt == 59:
                raise RuntimeError("The retrieval app did not start. Check the cell output for an error.")
            time.sleep(1)

    ui_url = output.eval_js(f"google.colab.kernel.proxyPort({port})")
    if not ui_url.endswith("/"):
        ui_url += "/"
    print(f"The retrieval app is using {config.get('device_label', torch.cuda.get_device_name(0))}.")
    print(f"Video folder: {video_dir}")
    display(HTML(f'<a href="{html.escape(ui_url, quote=True)}" target="_blank" rel="noreferrer" style="font-size:18px;font-weight:700">Open research interface</a>'))


launch_retrieval_app()
